# Case A — Biomedical KG (Gene / Tissue)
Mirrors `tests/golden/case_a` end-to-end: build a small Gene + Tissue graph, build CSR adjacency, run a 1-hop Expand from TP53, and join back to gene symbols.

In [ ]:
from pathlib import Path
import tempfile, pyarrow as pa
from caracaldb.onto.catalog import Catalog, save_catalog
from caracaldb.storage import create_bundle
from caracaldb.storage.node_store import open_node_store
from caracaldb.storage.edge_store import open_edge_store
from caracaldb.graph import build_csr
from caracaldb.graph.csr_reader import CsrReader

tmp = Path(tempfile.mkdtemp())
bundle = create_bundle(tmp / 'bio')
cat = Catalog.empty()
cat.register_class(iri='http://example.org/Gene', local_name='Gene')
save_catalog(bundle, cat)

genes = open_node_store(bundle, class_iri='http://example.org/Gene', local_name='Gene', create=True)
genes.append(pa.record_batch({'symbol': pa.array(['TP53','MDM2','BRCA1','EGFR','KRAS']),
                              'chromosome': pa.array(['17','12','17','7','12'])}))
iw = open_edge_store(bundle, property_iri='http://example.org/interactsWith',
                     local_name='interactsWith', create=True)
iw.append(pa.record_batch({'src': pa.array([0,0,1,2,3,4], type=pa.uint64()),
                           'dst': pa.array([1,2,3,0,4,0], type=pa.uint64())}))
csr_path = bundle.path / 'iw.csr'
build_csr(iw, num_vertices=5, out_path=csr_path, with_eids=True)
csr = CsrReader(csr_path)
print('Genes:', genes.num_rows, 'Edges:', iw.num_rows)


In [ ]:
from caracaldb.exec.expr import compile_expr
from caracaldb.exec.operator import run_pipeline
from caracaldb.exec.operators import (NodeScanOperator, FilterOperator,
                                       ExpandOperator, HashJoinOperator)

scan = NodeScanOperator(genes)
tp53 = FilterOperator(scan, compile_expr(('eq', ('col', 'symbol'), ('lit', 'TP53'))))
expand = ExpandOperator(tp53, forward=csr, direction='out')
join = HashJoinOperator(NodeScanOperator(genes), expand, build_key='nid', probe_key='dst')
out = pa.Table.from_batches(list(run_pipeline(join)))
print('TP53 neighbours:', out['symbol'].to_pylist())
